# Probabilistic Explanations for Tabular Data with `PyProbXAI`

This tutorial demonstrates how to compute and visualize **probable explanations** for tabular classifiers across three model paradigms:
1. **Decision Trees** — Exact & Greedy Probable Explanations (*UAI 2023*)
2. **Tree Ensembles (Random Forests)** — $k$-Pseudo-Probable & Majoritary Explanations (*ECML PKDD 2026*)
3. **Black-Box Models (Neural Networks)** — BanditExplain $(\eta, \delta)$-PAC Explanations


### The Tabular Explainability Challenge

Standard feature-attribution methods (e.g. SHAP values, feature importance) output continuous scores across all features, but cannot answer critical questions for algorithmic fairness and compliance:
- **Sufficiency**: *Is this subset of features sufficient on its own to guarantee the prediction?*
- **Invariance**: *Can unselected features change arbitrarily without affecting the decision?*

`PyProbXAI` identifies a minimal set of $k$ feature constraints with certified statistical precision:
$$\text{Precision}(S) = 1 - \text{Error}(S) \ge 1 - \epsilon$$

This certifies that for any individual satisfying these conditions, the classifier will predict the target class with probability at least $1 - \epsilon$ under the input distribution.

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# Ensure PyProbXAI is on path
sys.path.insert(0, os.path.abspath(".."))
import PyProbXAI as pe
from PyProbXAI import DecisionTreeExplainer, TreeEnsembleExplainer, BlackBoxExplainer


## 1. Dataset: UCI Adult Income Benchmark

We predict whether an individual's annual income exceeds $50K (`>50K` vs `<=50K`).
We preserve categorical feature mappings to ensure interpretable textual labels in explanations.


In [ ]:
data_path = os.path.join("..", "data", "adult.csv")
df = pd.read_csv(data_path)
X_raw, y = df.drop(columns="income"), df["income"]

categorical_cols = list(X_raw.select_dtypes(include=["object", "string"]).columns)
decoders = {}
X = X_raw.copy()

for c in categorical_cols:
    X[c], cat_names = pd.factorize(X[c], sort=True)
    decoders[X.columns.get_loc(c)] = list(cat_names)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y
)
X_raw_test = X_raw.iloc[X_test.index]

print(f"Loaded Adult dataset: {len(X_train)} train, {len(X_test)} test instances.")
print(f"Features ({len(X.columns)} total): {list(X.columns)}")


## 2. Decision Tree Explanations (UAI 2023)

For single decision trees, `PyProbXAI` computes exact and greedy bounds by directly traversing the branch partitions.


In [ ]:
dt = DecisionTreeClassifier(max_depth=8, random_state=0).fit(X_train, y_train)
print(f"Decision Tree Test Accuracy: {dt.score(X_test, y_test):.2%}")

dt_explainer = DecisionTreeExplainer(dt, feature_names=X.columns)


In [ ]:
instance_idx = 0
x_row = X_test.iloc[instance_idx]
true_label = y_test.iloc[instance_idx]

# Greedy descent (GD) explanation of size at most K=3
e_dt = dt_explainer.explain(x_row.values, k=3, method="gd")

# Text summary of the explainaiton
print(e_dt)

# Interactive view showing instance values and certified constraints
e_dt.show_tabular(x_row, title=f"Adult Individual #{instance_idx} (Decision Tree)", true_class=true_label)


## 3. Tree Ensemble & Random Forest Explanations (ECML PKDD 2026)

For forests and ensembles, `TreeEnsembleExplainer` provides two complementary explanation concepts:
1. **$k$-Pseudo-Probable Explanations**: A minimal subset of size $\le k$ certified across the ensemble.
2. **Majoritary Explanations**: A subset of conditions that guarantees at least $\ge 50\%$ of all trees in the ensemble vote for the target class with zero error.


In [ ]:
rf = RandomForestClassifier(n_estimators=20, max_depth=6, random_state=0).fit(X_train, y_train)
print(f"Random Forest Test Accuracy: {rf.score(X_test, y_test):.2%}")

rf_explainer = TreeEnsembleExplainer(rf, feature_names=X.columns)


In [ ]:
# 1. k-pseudo-probable explanation (GD)
e_rf_gd = rf_explainer.explain(x_row.values, k=4, method="gd")
print(e_rf_gd)
e_rf_gd.show_tabular(x_row, title=f"Adult Individual #{instance_idx} (Random Forest k-pseudo)", true_class=true_label)


In [ ]:
# 2. Majoritary explanation (UAI Majority consensus)
e_rf_maj = rf_explainer.explain(x_row.values, method="majority")
e_rf_maj.show_tabular(x_row, title=f"Adult Individual #{instance_idx} (Random Forest Majority)", true_class=true_label)


## 4. Black-Box Neural Network Explanations (BanditExplain)

For arbitrary non-linear classifiers, `BlackBoxExplainer` uses multi-armed bandits to sample feature perturbations and compute $(\eta, \delta)$-PAC certified bounds.


In [ ]:
nn_pipeline = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(64,), max_iter=300, random_state=0)
)
nn_pipeline.fit(X_train, y_train)
print(f"Neural Network Test Accuracy: {nn_pipeline.score(X_test, y_test):.2%}")

bb_explainer = BlackBoxExplainer(
    nn_pipeline,
    X_train,
    categorical_features=categorical_cols,
    cat_decoders=decoders,
    n_bins=3,
    eta=0.10,
    delta=0.05,
    random_state=0,
)


In [ ]:
e_bb = bb_explainer.explain(x_row.values, k=4)
e_bb.show_tabular(x_row, title=f"Adult Individual #{instance_idx} (Neural Network BE)", true_class=true_label)


## 5. Cross-Model Comparison & Export

`PyProbXAI` provides a completely uniform API across all three paradigms:
- **Decision Tree**: Identifies critical branch thresholds with deterministic error bounds.
- **Random Forest**: Computes consensus rules across the ensemble.
- **Black-Box Neural Net**: Derives statistical $(\eta, \delta)$-PAC sufficiency.

Each explanation view can be exported directly as a standalone publication-grade HTML report.


In [ ]:
os.makedirs("../results", exist_ok=True)
report_file = "../results/tabular_adult_comparison.html"
e_bb.show_tabular(x_row, title="Adult Cross-Model Benchmark", true_class=true_label).save(report_file)
print(f"Exported standalone report: {report_file}")


## Summary

With `PyProbXAI`, tabular explanations provide:
1. **Certified Feature Bounds**: Exact constraints sufficient for the model's prediction.
2. **Quantified Invariance**: Formal guarantees that remaining features can vary freely.
3. **Unified API**: Seamlessly works across Decision Trees, Tree Ensembles, and Black-Box Classifiers.
